# 🎮 CS2 AI Video Commentator
**Model:** InternVL2-8B + YOLO CS2  
**Görev:** CS2 video kliplerini otomatik analiz et — harita, kill tespiti, TR/EN açıklama  

---
### ⚙️ Kurulum
1. Runtime → Change Runtime Type → **A100 GPU**
2. Tümünü Çalıştır

In [ ]:
# ============================================================
# HÜCRE 1 — Kurulum
# ============================================================
!pip install -q transformers==4.40.0 accelerate einops timm sentencepiece
!pip install -q opencv-python-headless pillow pandas openpyxl
!pip install -q nltk rouge-score deep-translator gradio
!pip install -q ultralytics roboflow
!apt-get install -y ffmpeg -qq

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print('✅ Kurulum tamamlandı')

In [ ]:
# ============================================================
# HÜCRE 2 — GPU Kontrolü + Drive Bağlantısı
# ============================================================
import torch, os
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

from google.colab import drive
drive.mount('/content/drive')

VIDEO_DIR  = '/content/drive/MyDrive/CS2_Video_Caption/CS-2_Videolar'
CLEAN_DIR  = '/content/drive/MyDrive/CS2_Video_Caption/CS-2_Temiz'
EXCEL_PATH = '/content/drive/MyDrive/CS2_Video_Caption/cs-2.xlsx'
OUTPUT_DIR = '/content/drive/MyDrive/CS2_Video_Caption/results'

os.makedirs(CLEAN_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("✅ Path'ler hazır")

In [ ]:
# ============================================================
# HÜCRE 3 — Video Temizleme (Son 10 saniyeyi kes)
# ============================================================
import subprocess

def trim_video(src, dst, cut_last=10):
    probe = subprocess.run([
        'ffprobe','-v','error',
        '-show_entries','format=duration',
        '-of','default=noprint_wrappers=1:nokey=1', src
    ], capture_output=True, text=True)
    duration = float(probe.stdout.strip())
    keep = max(duration - cut_last, 1.0)
    subprocess.run([
        'ffmpeg','-y','-i', src,
        '-t', str(keep),
        '-c', 'copy', dst
    ], check=True, capture_output=True)
    return keep

for f in sorted(os.listdir(VIDEO_DIR)):
    if not f.endswith('.mp4'): continue
    src = os.path.join(VIDEO_DIR, f)
    dst = os.path.join(CLEAN_DIR, f)
    if os.path.exists(dst):
        print(f'⏭ zaten var: {f}')
        continue
    kept = trim_video(src, dst)
    print(f'✅ {f} → {kept:.1f}sn')

print('✅ Tüm videolar temizlendi')

In [ ]:
# ============================================================
# HÜCRE 4 — YOLO CS2 Player Modeli + Frame Extraction
# ============================================================
from ultralytics import YOLO
from huggingface_hub import hf_hub_download
from google.colab import userdata
import cv2, numpy as np
from PIL import Image

# HuggingFace token ile indir
# Colab Sol menü → 🔑 → HF_TOKEN ekle
import os
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

model_path = hf_hub_download(
    repo_id  = 'Vombit/yolo12n_cs2',
    filename = 'yolo12n_cs2.pt',
    token    = os.environ['HF_TOKEN']
)
yolo_player = YOLO(model_path)
print(f'✅ YOLO Player: {yolo_player.names}')

def extract_highlight_frames(video_path, top_k=8):
    cap = cv2.VideoCapture(video_path)
    frames, scores = [], []
    prev_gray = None
    idx = 0
    while True:
        ret, frame = cap.read()
        if not ret: break
        idx += 1
        if idx % 10 != 0: continue
        frame = cv2.resize(frame, (640, 360))
        gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        score = 0
        if prev_gray is not None:
            score += float(cv2.absdiff(gray, prev_gray).mean())
        results = yolo_player(frame, verbose=False)[0]
        score  += len(results.boxes) * 15
        frames.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
        scores.append(score)
        prev_gray = gray
    cap.release()
    if not frames: return []
    top_idx = np.argsort(scores)[-top_k:]
    top_idx.sort()
    return [frames[i] for i in top_idx]

def detect_events(frames):
    events = []
    prev_count = 0
    for i, frame in enumerate(frames):
        frame_np = np.array(frame.resize((1280, 720)))
        results  = yolo_player(frame_np, verbose=False, imgsz=1280)[0]
        count    = len(results.boxes)
        if count > prev_count:
            events.append(f'Frame {i+1}: Enemy appeared ({count} player)')
        elif count < prev_count:
            events.append(f'Frame {i+1}: Enemy disappeared — possible kill')
        elif count > 0:
            events.append(f'Frame {i+1}: Enemy visible ({count} player)')
        prev_count = count
    if not events:
        events.append('No enemy detected — player repositioning')
    return events

print('✅ Frame extraction + event detection hazır')

In [ ]:
# ============================================================
# HÜCRE 5 — YOLO Silah Modeli (Fine-Tuned)
# ============================================================
# Bu hücre Roboflow'dan CS2 silah veri setini indirip
# YOLOv8n modelini AK-47 ve M4A4 tespiti için fine-tune eder.
# Eğer model zaten eğitildiyse sadece yükler.

from ultralytics import YOLO
import os

WEAPON_MODEL_PATH = '/content/drive/MyDrive/CS2_Video_Caption/cs2_weapon_best.pt'

if os.path.exists(WEAPON_MODEL_PATH):
    # Model zaten eğitilmiş, direkt yükle
    yolo_weapon = YOLO(WEAPON_MODEL_PATH)
    print(f'✅ Silah modeli yüklendi: {yolo_weapon.names}')
else:
    # Fine-tune et
    print('Fine-tuning başlıyor...')
    from roboflow import Roboflow
    import requests
    from io import BytesIO

    # Roboflow'dan veri seti indir
    !wget -q "https://app.roboflow.com/ds/snPtJzcKko?key=EgZJBBES71" -O /content/cs2_dataset.zip
    !unzip -q -o /content/cs2_dataset.zip -d /content/cs2_dataset

    yolo_weapon = YOLO('yolov8n.pt')
    yolo_weapon.train(
        data    = '/content/cs2_dataset/data.yaml',
        epochs  = 30,
        imgsz   = 640,
        batch   = 16,
        device  = 0,
        name    = 'cs2_weapon',
        verbose = False
    )
    # Drive'a kaydet (bir daha eğitmemek için)
    import shutil
    shutil.copy('runs/detect/cs2_weapon/weights/best.pt', WEAPON_MODEL_PATH)
    yolo_weapon = YOLO(WEAPON_MODEL_PATH)
    print(f'✅ Fine-tuning tamamlandı: {yolo_weapon.names}')

def detect_weapon_yolo(frames):
    from collections import Counter
    votes = []
    for frame in frames:
        results = yolo_weapon(np.array(frame), verbose=False)[0]
        for box in results.boxes:
            cls_name = yolo_weapon.names[int(box.cls[0])]
            conf     = float(box.conf[0])
            if cls_name in ['AK-47', 'M4A4'] and conf > 0.25:
                votes.append(cls_name)
    return Counter(votes).most_common(1)[0][0] if votes else 'Unknown'

In [ ]:
# ============================================================
# HÜCRE 6 — InternVL2-8B Yükleme
# ============================================================
from transformers import AutoTokenizer, AutoModel
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode

model_name = 'OpenGVLab/InternVL2-8B'
tokenizer  = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
vlm        = AutoModel.from_pretrained(
    model_name,
    torch_dtype       = torch.bfloat16,
    device_map        = 'auto',
    trust_remote_code = True
).eval()

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

def frames_to_pixel_values(frames):
    transform = T.Compose([
        T.Lambda(lambda img: img.convert('RGB')),
        T.Resize((448, 448), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])
    return torch.stack([transform(f) for f in frames]).to(torch.bfloat16).cuda()

print('✅ InternVL2-8B hazır')
print(f'VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

In [ ]:
# ============================================================
# HÜCRE 7 — Veri Seti + Few-Shot Havuzu
# ============================================================
import pandas as pd

df = pd.read_excel(EXCEL_PATH)
print(f'Kolonlar: {df.columns.tolist()}')
print(f'Toplam video: {len(df)}')

few_shot_pool = []
for _, row in df.iterrows():
    vpath = os.path.join(CLEAN_DIR, row['Video adları'])
    if not os.path.exists(vpath): continue
    frames = extract_highlight_frames(vpath, top_k=4)
    if not frames: continue
    few_shot_pool.append({
        'frames'     : frames,
        'description': row['Video Açıklaması']
    })

print(f'✅ Few-shot havuzu: {len(few_shot_pool)} video')

In [ ]:
# ============================================================
# HÜCRE 8 — Ana Pipeline
# ============================================================
from deep_translator import GoogleTranslator
import random

def analyze_video(video_path, n_shots=3):
    frames = extract_highlight_frames(video_path, top_k=6)
    if not frames:
        return {'en': '', 'tr': '', 'map': '?', 'weapon': '?', 'kill': False}

    pixel_values = frames_to_pixel_values(frames)

    # ADIM 1: Harita tespiti
    map_pv = frames_to_pixel_values([frames[0]])
    with torch.no_grad():
        detected_map = vlm.chat(
            tokenizer         = tokenizer,
            pixel_values      = map_pv,
            question          = "<image>\nWhich CS2 map is this: Mirage, Dust2, Inferno, Nuke, Ancient, Anubis? One word only.",
            generation_config = dict(max_new_tokens=5, do_sample=False)
        ).strip()
    torch.cuda.empty_cache()

    # ADIM 2: Silah tespiti (YOLO)
    detected_weapon = detect_weapon_yolo(frames)

    # ADIM 3: Kill tespiti (YOLO Player)
    events        = detect_events(frames)
    events_text   = '\n'.join(events)
    kill_happened = any('possible kill' in e for e in events)
    kill_text     = 'A kill occurred.' if kill_happened else 'No kill detected.'

    # ADIM 4: Few-shot örnekler
    shot_text = ''
    if few_shot_pool and n_shots > 0:
        shots = random.sample(few_shot_pool, min(n_shots, len(few_shot_pool)))
        shot_text = 'Example descriptions:\n'
        for s in shots:
            shot_text += f'- {s["description"]}\n'
        shot_text += '\n'

    # ADIM 5: VLM açıklama
    prompt = '<image>\n' * len(frames)
    prompt += (
        f'You are a CS2 esports commentator.\n'
        f'Map: {detected_map}\n'
        f'Weapon: {detected_weapon}\n'
        f'YOLO Events:\n{events_text}\n'
        f'{kill_text}\n'
        f'{shot_text}'
        f'Write exactly 2 natural sentences. '
        f'Mention map area, weapon, and kill. '
        f'Start with: On {detected_map}...'
    )

    with torch.no_grad():
        en = vlm.chat(
            tokenizer         = tokenizer,
            pixel_values      = pixel_values,
            question          = prompt,
            generation_config = dict(max_new_tokens=100, do_sample=False)
        )
    torch.cuda.empty_cache()

    try:
        tr = GoogleTranslator(source='en', target='tr').translate(en.strip())
    except:
        tr = en.strip()

    return {
        'en'    : en.strip(),
        'tr'    : tr,
        'map'   : detected_map,
        'weapon': detected_weapon,
        'kill'  : kill_happened
    }

# Test
test_path = os.path.join(CLEAN_DIR, df.iloc[0]['Video adları'])
out = analyze_video(test_path)
print(f'🗺️  Harita:  {out["map"]}')
print(f'🔫 Silah:   {out["weapon"]}')
print(f'💀 Kill:    {out["kill"]}')
print(f'🇬🇧 EN: {out["en"]}')
print(f'🇹🇷 TR: {out["tr"]}')

In [ ]:
# ============================================================
# HÜCRE 9 — Tüm Veri Seti + ROUGE Değerlendirme
# ============================================================
from rouge_score import rouge_scorer as rs
import time

scorer  = rs.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
results = []

def safe_translate(text, src='tr', tgt='en', retries=3):
    for i in range(retries):
        try:
            time.sleep(1)
            return GoogleTranslator(source=src, target=tgt).translate(text)
        except Exception as e:
            print(f'  ⚠ ({i+1}/{retries}): {e}')
            time.sleep(3)
    return text

for _, row in df.iterrows():
    vpath = os.path.join(CLEAN_DIR, row['Video adları'])
    if not os.path.exists(vpath):
        print(f'⚠ atlandı: {row["Video adları"]}')
        continue

    print(f'⏳ {row["Video adları"]}')
    try:
        out   = analyze_video(vpath, n_shots=3)
        gt_en = safe_translate(row['Video Açıklaması'], src='tr', tgt='en')
        score = scorer.score(gt_en, out['en'])

        results.append({
            'video'  : row['Video adları'],
            'gt_tr'  : row['Video Açıklaması'],
            'pred_tr': out['tr'],
            'pred_en': out['en'],
            'map'    : out['map'],
            'weapon' : out['weapon'],
            'kill'   : out['kill'],
            'rouge1' : round(score['rouge1'].fmeasure, 4),
            'rouge2' : round(score['rouge2'].fmeasure, 4),
            'rougeL' : round(score['rougeL'].fmeasure, 4),
        })
        print(f'  ✅ ROUGE-L: {score["rougeL"].fmeasure:.3f} | {out["map"]} | {out["weapon"]}')

    except Exception as e:
        print(f'  ❌ hata: {e}')
        results.append({
            'video': row['Video adları'], 'gt_tr': row['Video Açıklaması'],
            'pred_tr': '', 'pred_en': '', 'map': '', 'weapon': '',
            'kill': False, 'rouge1': 0, 'rouge2': 0, 'rougeL': 0
        })

results_df = pd.DataFrame(results)
results_df.to_excel(os.path.join(OUTPUT_DIR, 'final_results.xlsx'), index=False)
print(f'\n📊 Toplam: {len(results_df)} video')
print(f'📊 ROUGE-1:  {results_df["rouge1"].mean():.3f}')
print(f'📊 ROUGE-2:  {results_df["rouge2"].mean():.3f}')
print(f'📊 ROUGE-L:  {results_df["rougeL"].mean():.3f}')

In [ ]:
# ============================================================
# HÜCRE 10 — Sonuç Grafikleri
# ============================================================
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('CS2 VLM Pipeline — Final Sonuçlar', fontsize=14, fontweight='bold')

axes[0].hist(results_df['rougeL'], bins=10, color='#2ecc71', edgecolor='black', alpha=0.85)
axes[0].axvline(results_df['rougeL'].mean(), color='red', linestyle='--',
                label=f'Ort: {results_df["rougeL"].mean():.3f}')
axes[0].set_xlabel('ROUGE-L')
axes[0].set_ylabel('Video Sayısı')
axes[0].set_title('ROUGE-L Dağılımı')
axes[0].legend()

metrics = ['rouge1', 'rouge2', 'rougeL']
means   = [results_df[m].mean() for m in metrics]
colors  = ['#3498db', '#e74c3c', '#2ecc71']
bars    = axes[1].bar(metrics, means, color=colors, edgecolor='black')
for bar, val in zip(bars, means):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                 f'{val:.3f}', ha='center', fontweight='bold')
axes[1].set_title('Metrik Karşılaştırması')
axes[1].set_ylim(0, 0.5)

plt.tight_layout()
chart_path = os.path.join(OUTPUT_DIR, 'final_chart.png')
plt.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Grafik kaydedildi')

In [ ]:
# ============================================================
# HÜCRE 11 — Gradio Demo
# ============================================================
import gradio as gr
import tempfile

def gradio_fn(video):
    if video is None:
        return 'Video yüklenmedi.', 'No video.', '?', '?', '?'
    try:
        tmp = tempfile.mktemp(suffix='.mp4')
        trim_video(video, tmp, cut_last=10)
        out = analyze_video(tmp, n_shots=3)
        kill_str = '✅ Kill tespit edildi' if out['kill'] else '❌ Kill tespit edilmedi'
        return out['tr'], out['en'], out['map'], out['weapon'], kill_str
    except Exception as e:
        return f'Hata: {e}', f'Error: {e}', '?', '?', '?'

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎮 CS2 AI Video Commentator")
    gr.Markdown("CS2 highlight klibini yükle — harita, silah ve kill otomatik analiz edilsin.")

    with gr.Row():
        with gr.Column(scale=1):
            video_input = gr.Video(label="📁 CS2 Video Yükle (.mp4)")
            submit_btn  = gr.Button("🔍 Analiz Et", variant="primary")

        with gr.Column(scale=2):
            with gr.Row():
                map_out    = gr.Textbox(label="🗺️ Harita",  scale=1)
                weapon_out = gr.Textbox(label="🔫 Silah",   scale=1)
                kill_out   = gr.Textbox(label="💀 Kill",    scale=1)
            tr_out = gr.Textbox(label="🇹🇷 Türkçe Açıklama",     lines=8)
            en_out = gr.Textbox(label="🇬🇧 English Description",  lines=8)

    submit_btn.click(
        fn      = gradio_fn,
        inputs  = [video_input],
        outputs = [tr_out, en_out, map_out, weapon_out, kill_out]
    )

demo.launch(share=True, debug=False)